## Examples from *lme4: Mixed-effects Modeling with R* (Bates, 2010)

Walks through the main worked examples in `example/data/lMMwR.pdf` to
demonstrate `lmpy.lme`. Numerical outputs are pinned in
`tests/test_lme_examples.py`.

The model API matches lme4's: `formula` uses bar syntax (`(1|g)` scalar,
`(1+x|g)` correlated vector, `(0+x|g)` zero-intercept slope, `(1|a) +
(1|a:b)` nested), `REML=True` is the default, and printed numerics are
the standard ML/REML summary.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2

from lmpy.lme import lme

DATA = Path('..').resolve() / 'datasets'
def D(pkg, name):
    return pd.read_csv(DATA / pkg / f'{name}.csv')

### Ch 1 § 1.4 — `Dyestuff`: a single random factor

In [ ]:
Dyestuff = D('lme4', 'Dyestuff')
fm01 = lme('Yield ~ 1 + (1|Batch)', Dyestuff)              # REML default
print(fm01)
print('REML criterion:', fm01.REML_criterion)

Same fit by ML — note the deviance / AIC / BIC summary the book prints in Table 1.4.

In [ ]:
fm01ML = lme('Yield ~ 1 + (1|Batch)', Dyestuff, REML=False)
print(fm01ML)
print(f'AIC={fm01ML.AIC:.4f}  BIC={fm01ML.BIC:.4f}  '
      f'logLik={fm01ML.loglike:.4f}  deviance={fm01ML.deviance:.4f}  '
      f'df.resid={fm01ML.df_resid}')

`Dyestuff2` (§ 1.5) — singular fit: REML drives σ̂_Batch to 0.

In [ ]:
Dyestuff2 = D('lme4', 'Dyestuff2')
fm02 = lme('Yield ~ 1 + (1|Batch)', Dyestuff2)
print(fm02)

### Ch 2 § 2.1 — `Penicillin`: crossed random factors

In [ ]:
Penicillin = D('lme4', 'Penicillin')
fm03 = lme('diameter ~ 1 + (1|plate) + (1|sample)', Penicillin)
print(fm03)
print('n =', fm03.n, ' n_groups =', fm03.n_groups)

### Ch 2 § 2.2 — `Pastes`: nested random factors and an LRT for σ_batch = 0

`fm04` includes both `(1|sample)` and `(1|batch)`. The reduced model
`fm04a` drops `(1|batch)`. Bates uses `anova(fm04a, fm04)` to test
whether σ_batch is non-zero (book § 2.2.4).

In [ ]:
Pastes = D('lme4', 'Pastes')
fm04  = lme('strength ~ 1 + (1|sample) + (1|batch)', Pastes, REML=False)
fm04a = lme('strength ~ 1 + (1|sample)',             Pastes, REML=False)
print(fm04)
print()
print(fm04a)

In [ ]:
chisq = fm04a.deviance - fm04.deviance
df    = fm04.npar - fm04a.npar
p     = chi2.sf(chisq, df)
print(f'LRT fm04a vs fm04:  chisq={chisq:.4f}  df={df}  p={p:.4f}')

### Ch 3 § 3.2 — `sleepstudy`: random slopes

`fm07` is the **uncorrelated** version: `(1|Subject) +
(0+Days|Subject)`. Because the same group `Subject` appears in two
bars, the second one shows up under the disambiguated key
`Subject.1`.

In [ ]:
sleepstudy = D('lme4', 'sleepstudy')
fm07 = lme(
    'Reaction ~ 1 + Days + (1|Subject) + (0+Days|Subject)',
    sleepstudy, REML=False,
)
print(fm07)

`fm06` is the **correlated** version: `(1+Days|Subject)` — a single
vector bar of size 2, fitting an intercept SD, a slope SD, and one
correlation parameter.

In [ ]:
fm06 = lme('Reaction ~ 1 + Days + (1+Days|Subject)', sleepstudy, REML=False)
print(fm06)

LRT for the correlation: `fm06` adds 1 parameter (the off-diagonal of
the Cholesky factor) over `fm07`. Book result: χ²=0.0639, p=0.8004.

In [ ]:
chisq = fm07.deviance - fm06.deviance
df    = fm06.npar - fm07.npar
p     = chi2.sf(chisq, df)
print(f'LRT fm07 vs fm06:  chisq={chisq:.4f}  df={df}  p={p:.4f}')

### Ch 4 § 4.1 — `Machines`: fixed factor + nested random factor

In [ ]:
Machines = D('nlme', 'Machines')
fm10 = lme(
    'score ~ Machine + (1|Worker) + (1|Machine:Worker)',
    Machines, REML=False,
)
print(fm10)
print('n_groups:', fm10.n_groups)

### Ch 4 § 4.2 — `ergoStool`: random vs fixed treatment

`fm16` treats both `Subject` and `Type` as random. `fm17` keeps
`Subject` random but treats `Type` (4 levels) as a fixed effect — the
more conventional choice.

In [ ]:
ergoStool = D('nlme', 'ergoStool')
fm16 = lme('effort ~ 1 + (1|Subject) + (1|Type)', ergoStool, REML=False)
print(fm16)

In [ ]:
fm17 = lme('effort ~ 1 + Type + (1|Subject)', ergoStool, REML=False)
print(fm17)